# CV10 — Khôi phục nhiều loại suy giảm

Ảnh sạch y, ảnh suy giảm x có cùng tọa độ. Mô hình dự đoán residual, cộng vào x. Conditioning dự đoán scale/bias từ feature toàn ảnh, tương tự FiLM.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Ảnh sạch y, ảnh suy giảm x có cùng tọa độ. Mô hình dự đoán residual, cộng vào x. Conditioning dự đoán scale/bias từ feature toàn ảnh, tương tự FiLM. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv10_restoration import Restoration, loss_fn
clean_images = torch.rand(2, 3, 32, 32, device=device)
degraded = (clean_images + 0.1 * torch.randn_like(clean_images)).clamp(0, 1)
model = Restoration(d=32, condition=True).to(device)
batch = (degraded, clean_images)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    restored = model(degraded)
print('Input/output:', degraded.shape, restored.shape)
print('Identity baseline MSE:', F.mse_loss(degraded, clean_images).item())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Identity → residual CNN → feature conditioning → Restormer/PromptIR là hướng đọc thêm, chưa tái lập trong module.


In [ ]:
model.eval()
with torch.no_grad():
    restored = model(degraded).clamp(0, 1)
    baseline_mse = (degraded - clean_images).square().mean((1, 2, 3))
    restored_mse = (restored - clean_images).square().mean((1, 2, 3))
print('Fixture identity PSNR:', (-10 * baseline_mse.clamp_min(1e-12).log10()).mean().item())
print('Fixture model PSNR:', (-10 * restored_mse.clamp_min(1e-12).log10()).mean().item())
print('Compare condition=False/True, then try a trained multi-scale restoration architecture.')


## Thử nghiệm tiếp theo

Identity → residual CNN → feature conditioning → Restormer/PromptIR là hướng đọc thêm, chưa tái lập trong module.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv10_restoration.py) và [tài liệu gốc](../REFERENCE.md).
